# Détection de Fraude Transactionnelle — Modèle à Risque Limité (AI Act)

⚠️ **Risque limité** — Art. 50 | Obligations de transparence pour les systèmes de décision automatisée

## 0. Dépendances optionnelles

In [ ]:
# Author: Octo Technology MLOps Tribe
%pip install shap pandera --quiet

## 1. Imports

In [ ]:
# Author: Octo Technology MLOps Tribe
import json
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import pandera as pa
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
ARTIFACTS_DIR = Path(tempfile.mkdtemp())
print(f"Répertoire artefacts temporaires : {ARTIFACTS_DIR}")

## 2. Configuration MLflow

In [ ]:
# Author: Octo Technology MLOps Tribe
PROJECT_NAME = "Fraud-Detection-Payments"
MLFLOW_TRACKING_URI = f"http://model-platform.com/registry/{PROJECT_NAME}/"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("transaction_fraud_detection")
print(f"MLflow tracking URI : {MLFLOW_TRACKING_URI}")

## 3. Génération des données synthétiques

In [ ]:
# Author: Octo Technology MLOps Tribe
np.random.seed(42)
N = 8000

transaction_amount      = np.random.exponential(scale=80, size=N).clip(1, 5000).round(2)
merchant_category       = np.random.randint(0, 10, N)
transaction_hour        = np.random.randint(0, 24, N)
day_of_week             = np.random.randint(0, 7, N)
distance_from_home      = np.random.exponential(scale=30, size=N).clip(0, 500).round(1)
distance_from_last_txn  = np.random.exponential(scale=20, size=N).clip(0, 300).round(1)
ratio_to_median_amount  = np.random.exponential(scale=1.2, size=N).clip(0.1, 20).round(3)
is_international        = (np.random.rand(N) < 0.15).astype(int)
num_transactions_24h    = np.random.randint(1, 30, N)
time_since_last_txn     = np.random.exponential(scale=120, size=N).clip(1, 1440).round(0)

# Probabilité de fraude (modèle logistique)
log_odds = (
    -5.0
    + 0.003 * transaction_amount
    + 0.04 * distance_from_home
    + 0.06 * distance_from_last_txn
    + 0.8 * ratio_to_median_amount
    + 1.2 * is_international
    + 0.04 * num_transactions_24h
    - 0.001 * time_since_last_txn
    + 0.3 * (transaction_hour < 6).astype(int)
)
prob_fraud = 1 / (1 + np.exp(-log_odds))
is_fraud = (np.random.rand(N) < prob_fraud).astype(int)

FEATURES = [
    "transaction_amount", "merchant_category", "transaction_hour", "day_of_week",
    "distance_from_home", "distance_from_last_txn", "ratio_to_median_amount",
    "is_international", "num_transactions_24h", "time_since_last_txn",
]
TARGET = "is_fraud"
PROTECTED_ATTRIBUTES = ["is_international"]

df = pd.DataFrame({
    "transaction_amount":     transaction_amount,
    "merchant_category":      merchant_category,
    "transaction_hour":       transaction_hour,
    "day_of_week":            day_of_week,
    "distance_from_home":     distance_from_home,
    "distance_from_last_txn": distance_from_last_txn,
    "ratio_to_median_amount": ratio_to_median_amount,
    "is_international":       is_international,
    "num_transactions_24h":   num_transactions_24h,
    "time_since_last_txn":    time_since_last_txn,
    "is_fraud":               is_fraud,
})
print(f"Dataset : {len(df):,} lignes | Taux de fraude : {df['is_fraud'].mean():.1%}")

## 4. Validation Pandera & statistiques descriptives

In [ ]:
# Author: Octo Technology MLOps Tribe
FRAUD_SCHEMA = pa.DataFrameSchema(
    name="transaction_fraud_input_schema",
    description="Contrat de données du modèle de détection de fraude — AI Act Art. 50",
    columns={
        "transaction_amount":     pa.Column(float, checks=pa.Check.in_range(0, 100_000), nullable=False, description="Montant de la transaction (€)"),
        "merchant_category":      pa.Column(int,   checks=pa.Check.in_range(0, 9),       nullable=False, description="Catégorie de marchand (0-9)"),
        "transaction_hour":       pa.Column(int,   checks=pa.Check.in_range(0, 23),      nullable=False, description="Heure de la transaction (0-23)"),
        "day_of_week":            pa.Column(int,   checks=pa.Check.in_range(0, 6),       nullable=False, description="Jour de la semaine (0=lundi)"),
        "distance_from_home":     pa.Column(float, checks=pa.Check.in_range(0, 10_000),  nullable=False, description="Distance du domicile (km)"),
        "distance_from_last_txn": pa.Column(float, checks=pa.Check.in_range(0, 10_000),  nullable=False, description="Distance depuis la dernière transaction (km)"),
        "ratio_to_median_amount": pa.Column(float, checks=pa.Check.in_range(0, 1_000),   nullable=False, description="Ratio montant / médiane client"),
        "is_international":       pa.Column(int,   checks=pa.Check.isin([0, 1]),          nullable=False, description="Transaction internationale — ⚠️ attribut sensible (géographie)"),
        "num_transactions_24h":   pa.Column(int,   checks=pa.Check.in_range(1, 1_000),   nullable=False, description="Nombre de transactions dans les 24h"),
        "time_since_last_txn":    pa.Column(float, checks=pa.Check.in_range(0, 10_000),  nullable=False, description="Temps depuis la dernière transaction (minutes)"),
        "is_fraud":               pa.Column(int,   checks=pa.Check.isin([0, 1]),          nullable=False, description="Variable cible : 1 = fraude, 0 = transaction légitime"),
    },
    coerce=False,
    strict=True,
)

try:
    FRAUD_SCHEMA.validate(df, lazy=True)
    PANDERA_STATUS = "PASS"
    PANDERA_ERRORS = 0
    print("✅ Validation Pandera : SUCCÈS")
except pa.errors.SchemaErrors as exc:
    PANDERA_STATUS = "FAIL"
    PANDERA_ERRORS = len(exc.failure_cases)
    print(f"❌ Validation Pandera : ÉCHEC ({PANDERA_ERRORS} erreurs)")

In [ ]:
# Author: Octo Technology MLOps Tribe
print("=== Statistiques descriptives ===")
display(df[FEATURES + [TARGET]].describe().round(3))

print("\n=== Valeurs manquantes ===")
missing = df.isnull().sum()
print(missing[missing > 0].to_string() if missing.any() else "Aucune valeur manquante.")

print(f"\n=== Distribution de la variable cible ===")
print(f"Légitime (0) : {(df[TARGET] == 0).sum():,} ({(df[TARGET] == 0).mean():.1%})")
print(f"Fraude   (1) : {(df[TARGET] == 1).sum():,} ({(df[TARGET] == 1).mean():.1%})")

INT_LABELS = ["Domestique", "International"]
int_dist = df["is_international"].value_counts().sort_index()
print("\n=== Distribution par zone géographique (attribut sensible) ===")
for k, count in int_dist.items():
    fr = df.loc[df["is_international"] == k, TARGET].mean()
    print(f"  {INT_LABELS[k]:<15} : {count:>5} obs | taux de fraude : {fr:.1%}")

In [ ]:
# Author: Octo Technology MLOps Tribe
SCHEMA_YAML_EXPORTED = False
try:
    schema_yaml_path = ARTIFACTS_DIR / "pandera_schema.yaml"
    with open(schema_yaml_path, "w") as f:
        f.write(FRAUD_SCHEMA.to_yaml())
    SCHEMA_YAML_EXPORTED = True
    print("Schéma Pandera exporté → pandera_schema.yaml")
except Exception as e:
    print(f"Export YAML non disponible : {e}")

validation_report = {
    "schema_name": FRAUD_SCHEMA.name,
    "validation_status": PANDERA_STATUS,
    "validation_errors": PANDERA_ERRORS,
    "n_rows_validated": int(len(df)),
    "protected_attributes": PROTECTED_ATTRIBUTES,
    "target_distribution": {
        "class_0_legitime": int((df[TARGET] == 0).sum()),
        "class_1_fraude":   int((df[TARGET] == 1).sum()),
        "fraud_rate":       round(float(df[TARGET].mean()), 4),
    },
}
validation_report_path = ARTIFACTS_DIR / "data_validation_report.json"
with open(validation_report_path, "w") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)
print("Rapport de validation exporté → data_validation_report.json")

## 5. Prétraitement

In [ ]:
# Author: Octo Technology MLOps Tribe
X, y = df[FEATURES], df[TARGET]

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42, stratify=y_train_full)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=FEATURES)
X_val_sc   = pd.DataFrame(scaler.transform(X_val),       columns=FEATURES)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=FEATURES)

print(f"Train : {len(X_train):,}  |  Validation : {len(X_val):,}  |  Test : {len(X_test):,}")
print(f"Taux de fraude — train : {y_train.mean():.1%}  |  test : {y_test.mean():.1%}")

## 6. Entraînement du modèle

In [ ]:
# Author: Octo Technology MLOps Tribe
PARAMS = {
    "n_estimators":    200,
    "max_depth":       8,
    "min_samples_split": 10,
    "min_samples_leaf":  5,
    "class_weight":    "balanced",
    "random_state":    42,
    "n_jobs":          -1,
}

model = RandomForestClassifier(**PARAMS)
model.fit(X_train_sc, y_train)

val_auc = roc_auc_score(y_val, model.predict_proba(X_val_sc)[:, 1])
print(f"AUC-ROC validation : {val_auc:.4f}")
print("Entraînement terminé.")

## 7. Évaluation sur le jeu de test

In [ ]:
# Author: Octo Technology MLOps Tribe
y_pred  = model.predict(X_test_sc)
y_proba = model.predict_proba(X_test_sc)[:, 1]

METRICS = {
    "accuracy":  round(float(accuracy_score(y_test, y_pred)),  4),
    "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
    "recall":    round(float(recall_score(y_test, y_pred,    zero_division=0)), 4),
    "f1_score":  round(float(f1_score(y_test, y_pred,        zero_division=0)), 4),
    "auc_roc":   round(float(roc_auc_score(y_test, y_proba)), 4),
}

report_dict = classification_report(y_test, y_pred, target_names=["Légitime", "Fraude"], output_dict=True)
print(classification_report(y_test, y_pred, target_names=["Légitime", "Fraude"]))
print("\n".join(f"{k:<12}: {v}" for k, v in METRICS.items()))

In [ ]:
# Author: Octo Technology MLOps Tribe
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {METRICS['auc_roc']:.3f}")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.fill_between(fpr, tpr, alpha=0.08, color="steelblue")
ax.set_xlabel("Taux de faux positifs (FPR)")
ax.set_ylabel("Taux de vrais positifs (TPR)")
ax.set_title("Courbe ROC — Détection de Fraude Transactionnelle")
ax.legend()
plt.tight_layout()
roc_path = ARTIFACTS_DIR / "roc_curve.png"
plt.savefig(roc_path, dpi=150)
plt.show()

## 8. Analyse d'équité (Fairness)

In [ ]:
# Author: Octo Technology MLOps Tribe
df_eval = X_test.copy()
df_eval["is_fraud"]   = y_test.values
df_eval["predicted"]  = y_pred
df_eval["proba"]      = y_proba

def subgroup_metrics(group):
    if len(group) < 20:
        return None
    try:
        auc = round(float(roc_auc_score(group["is_fraud"], group["proba"])), 4) if group["is_fraud"].nunique() > 1 else "N/A"
    except Exception:
        auc = "N/A"
    return {
        "n":           int(len(group)),
        "fraud_rate":  round(float(group["is_fraud"].mean()), 4),
        "accuracy":    round(float(accuracy_score(group["is_fraud"], group["predicted"])), 4),
        "recall":      round(float(recall_score(group["is_fraud"], group["predicted"], zero_division=0)), 4),
        "precision":   round(float(precision_score(group["is_fraud"], group["predicted"], zero_division=0)), 4),
        "auc_roc":     auc,
    }

fairness_by_intl = {str(k): subgroup_metrics(v) for k, v in df_eval.groupby("is_international", observed=True)}

FAIRNESS_REPORT = {
    "protected_attributes": PROTECTED_ATTRIBUTES,
    "note": "Le critère 'is_international' est un attribut sensible — son utilisation doit être validée pour éviter une discrimination géographique.",
    "by_international": fairness_by_intl,
}

print("=== Performance par zone géographique ===")
print(pd.DataFrame(fairness_by_intl).T.to_string())

## 9. Explicabilité

In [ ]:
# Author: Octo Technology MLOps Tribe
importances = model.feature_importances_
fi_df = pd.DataFrame({"feature": FEATURES, "importance": importances}).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["#c0392b" if f in PROTECTED_ATTRIBUTES else "steelblue" for f in fi_df["feature"][::-1]]
ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1], color=colors)
ax.set_title("Importance des variables (Gini) — rouge = attribut protégé")
ax.set_xlabel("Importance relative")
plt.tight_layout()
fi_path = ARTIFACTS_DIR / "feature_importance.png"
plt.savefig(fi_path, dpi=150)
plt.show()
print(fi_df.to_string(index=False))

In [ ]:
# Author: Octo Technology MLOps Tribe
SHAP_AVAILABLE = False
try:
    import shap
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test_sc.iloc[:300])
    plt.figure()
    shap.summary_plot(shap_values[1], X_test_sc.iloc[:300], show=False, plot_size=(10, 6))
    shap_path = ARTIFACTS_DIR / "shap_summary.png"
    plt.savefig(shap_path, dpi=150, bbox_inches="tight")
    plt.close()
    SHAP_AVAILABLE = True
    print("SHAP summary plot généré.")
except ImportError:
    print("SHAP non disponible — installer avec : pip install shap")

## 10. Préparation des artefacts de documentation

In [ ]:
# Author: Octo Technology MLOps Tribe
cr_path = ARTIFACTS_DIR / "classification_report.json"
with open(cr_path, "w") as f:
    json.dump(report_dict, f, indent=2, ensure_ascii=False)

fi_csv_path = ARTIFACTS_DIR / "feature_importance.csv"
fi_df.to_csv(fi_csv_path, index=False)

fairness_path = ARTIFACTS_DIR / "fairness_report.json"
with open(fairness_path, "w") as f:
    json.dump(FAIRNESS_REPORT, f, indent=2, ensure_ascii=False)

pp_context = {"schema_name": FRAUD_SCHEMA.name, "pandera_status": PANDERA_STATUS, "pandera_errors": PANDERA_ERRORS}
preprocessing_md = Path("preprocessing_description_template.md").read_text(encoding="utf-8").format_map(pp_context)
pp_path = ARTIFACTS_DIR / "preprocessing_description.md"
pp_path.write_text(preprocessing_md, encoding="utf-8")

print("Artefacts préparés :")
for p in sorted(ARTIFACTS_DIR.iterdir()):
    print(f"  {p.name}")

## 11. Logging MLflow

In [ ]:
# Author: Octo Technology MLOps Tribe
MODEL_NAME  = "transaction_fraud_detector"
TEAM        = "mlops-tribe"
ENVIRONMENT = "staging"

model_card_text = Path("model_card.md").read_text(encoding="utf-8")
model_card_path = ARTIFACTS_DIR / "model_card.md"
model_card_path.write_text(model_card_text, encoding="utf-8")

with mlflow.start_run(run_name=f"{MODEL_NAME}_v1") as run:
    mlflow.log_params(PARAMS)
    mlflow.log_param("scaler",           "StandardScaler")
    mlflow.log_param("feature_count",    len(FEATURES))
    mlflow.log_param("features",         ", ".join(FEATURES))
    mlflow.log_param("stratified_split", True)
    mlflow.log_param("train_size",       len(X_train))
    mlflow.log_param("val_size",         len(X_val))
    mlflow.log_param("test_size",        len(X_test))

    mlflow.log_metrics(METRICS)
    mlflow.log_metric("auc_roc_validation",  round(val_auc, 4))
    mlflow.log_metric("fraud_rate_train",    round(float(y_train.mean()), 4))
    mlflow.log_metric("fraud_rate_test",     round(float(y_test.mean()), 4))
    mlflow.log_metric("dataset_total_size",  N)

    mlflow.set_tag("team",        TEAM)
    mlflow.set_tag("environment", ENVIRONMENT)
    mlflow.set_tag("model_type",  "RandomForestClassifier")
    mlflow.set_tag("framework",   "scikit-learn")

    mlflow.set_tag("data_source",            "Données synthétiques — distribution calibrée sur des données transactionnelles réelles")
    mlflow.set_tag("data_period",            "N/A — données synthétiques")
    mlflow.set_tag("data_language",          "N/A — données tabulaires")
    mlflow.set_tag("contains_personal_data", "non — données transactionnelles pseudonymisées, aucun identifiant client")
    mlflow.set_tag("dataset_size",           f"train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")
    mlflow.set_tag("protected_attributes",   ", ".join(PROTECTED_ATTRIBUTES))
    mlflow.set_tag("data_schema_name",       FRAUD_SCHEMA.name)
    mlflow.set_tag("data_schema_version",    "1.0")
    mlflow.set_tag("pandera_validation",     PANDERA_STATUS)

    mlflow.set_tag("threshold_accuracy",  "0.90")
    mlflow.set_tag("threshold_f1",        "0.60")
    mlflow.set_tag("threshold_auc_roc",   "0.85")
    mlflow.set_tag("threshold_recall",    "0.70")

    mlflow.set_tag("ai_act_risk_level",         "limité")
    mlflow.set_tag("ai_act_annex_iii_category", "Art. 50 — Obligations de transparence pour les systèmes de décision automatisée sur transactions financières")

    mlflow.set_tag("mlflow.note.content", model_card_text)

    mlflow.log_artifact(str(cr_path))
    mlflow.log_artifact(str(fi_csv_path))
    mlflow.log_artifact(str(fi_path))
    mlflow.log_artifact(str(fairness_path))
    mlflow.log_artifact(str(roc_path))
    mlflow.log_artifact(str(pp_path))
    mlflow.log_artifact(str(validation_report_path))
    if SCHEMA_YAML_EXPORTED:
        mlflow.log_artifact(str(schema_yaml_path))
    if SHAP_AVAILABLE:
        mlflow.log_artifact(str(shap_path))
    mlflow.log_artifact(str(model_card_path))

    signature     = infer_signature(X_train_sc, model.predict_proba(X_train_sc))
    input_example = X_train_sc.head(3)
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="custom_model",
        registered_model_name=MODEL_NAME,
        signature=signature,
        input_example=input_example,
    )
    run_id = run.info.run_id

print(f"\n✅ Run MLflow : {run_id}")
print(f"   Modèle enregistré : {MODEL_NAME}")
print(f"   AUC-ROC : {METRICS['auc_roc']}  |  F1 : {METRICS['f1_score']}  |  Accuracy : {METRICS['accuracy']}")